# Ten Thousand Dawns — a model that writes its own epic

*A reproducible music notebook: a small sequence model, trained here, composing the piece you hear.*

Most music you can call "generated" arrives as an mp3 from behind an API, and there is no way to ask
it what it did. This notebook is the opposite of that. It starts with nothing but numpy and the
Python standard library, invents a corpus out of music theory, trains a 19,975-parameter recurrent
network on that corpus with a forward and a backward pass written out by hand, samples three voices
from the trained weights, synthesises them into 170 seconds of stereo audio, and masters the result
to the broadcast loudness standard. Then it ships the weights beside the track, bound to it by a
hash, so anyone can reload them and get the same 30 MB of audio back byte for byte.

That last part is the point. A model you cannot re-run is a rumour. Everything here is a function of
one integer seed and 80 kB of float32, and the notebook proves it in section 12 by throwing the
trained model away, reading the published file back off disk, and rendering the piece again.

**What you will learn**

1. How to turn music into a token stream a sequence model can actually learn — a sixteenth-note
   grid, one chord token per bar, and a `TIE` token that gives notes their length.

2. How a GRU works from the inside: the update and reset gates, the candidate state, the blend that
   lets it hold a motif across a bar line, and the backward pass through all of it, checked against
   finite differences rather than hoped over.

3. Constrained sampling — how to let a small model choose freely while making every bar it emits
   well formed, so you are not quietly hand-writing the music yourself.

4. Additive synthesis, envelopes, a multi-tap reverb and a boxcar shelf, all in numpy, because the
   offline sandbox has no scipy, no librosa, no soundfile and no ffmpeg. The whole audio toolchain
   here is `numpy` plus the standard library's `wave`.

5. Mastering as arithmetic: ITU-R BS.1770-4 gated loudness, the EBU Tech 3342 loudness range, and
   why a peak-to-loudness ratio around 11 dB is the number to aim at instead of "as loud as
   possible".

**The shape of the piece.** Eight sections of eight bars: a bare statement of the hook, a verse that
brings in the bass, a build, the drop, a break that goes somewhere new, a second build, the return of
the hook a diatonic third higher, and a resolution that thins back out to where it started. That
arrangement is a human decision and it is written down as a table you can edit. Every note inside it
came out of the model.

In [ ]:
import hashlib, json, math, os, struct, wave
import numpy as np

# One seed for everything: the corpus grammar, the weight initialisation, the training batches, the
# sampling and the noise in the drums. Nothing here reads the clock, the OS entropy pool or the
# network, so two runs of this notebook produce the same 30 MB file.
SEED = 20260726
SR, BPM, SPB, BARS = 44100, 90, 16, 64
STEP = SR * 60 // (BPM * 4)           # 7350 samples per sixteenth note at 90 BPM: an exact integer,
BAR, NS = STEP * SPB, STEP * SPB * BARS       # so bar boundaries never drift off the sample grid
TAIL = 2 * SR                         # room for the reverb to decay past the final bar
os.makedirs("out", exist_ok=True)
print(f"{BARS} bars of 4/4 at {BPM} BPM = {SPB * BARS} grid steps = {NS} samples")
print(f"that is {NS / SR:.2f} s of {SR} Hz stereo, about the length of the epics this piece follows")

## 1 · An alphabet the model can spell music in

A sequence model needs a finite alphabet, and the choice of alphabet decides what the model is able
to learn. Ours has **31 tokens**, designed so that any string of them is playable music.

Twenty-one are **melodic degrees** — three octaves of a seven-note scale. Working in scale degrees
rather than semitones means the model cannot play a wrong note: the key is baked into the alphabet,
so the whole of its small capacity goes on rhythm, contour and phrase shape instead of spending most
of it rediscovering which twelve semitones belong to A natural minor. That is a deliberate trade. A
chromatic alphabet would let the model modulate and go outside the key; it would also need ten times
the parameters and a hundred times the data before it stopped sounding like a wrong-note generator.

Two tokens carry time. `REST` is silence for one sixteenth; `TIE` holds whatever is already sounding
for one more sixteenth. That pair is what gives notes *duration* without a separate duration
vocabulary — a dotted quarter note is one onset followed by five ties — and it means the grid is
strictly one token per sixteenth, which keeps every sequence aligned to the beat by construction.

The last eight are **chords**, one per bar, and they cost almost nothing to define. Stacking
alternate degrees of the scale gives the right chord quality for free: degree 0 comes out A minor,
degree 2 comes out C major, degree 5 comes out F major, degree 6 comes out G major, degree 1 comes
out B diminished. There is no chord table anywhere in this notebook. Token 7 is the one hand-made
exception, a tonic suspended fourth, kept because a build wants a chord that refuses to resolve.

In [ ]:
# Every pitch in the piece is an index into these seven intervals, so nothing can land outside the key.
SCALE = (0, 2, 3, 5, 7, 8, 10)        # A natural minor: the interval pattern of the whole piece
REST, TIE, P0, NPITCH = 0, 1, 2, 21   # rest, sustain, then 21 melodic degrees starting at token 2
C0, NCHORD = P0 + NPITCH, 8           # then eight chord tokens, one of which is played per bar
VOCAB = C0 + NCHORD
MEL_ROOT, BASS_ROOT, PAD_ROOT = 57, 33, 45      # MIDI A3, A1 and A2: melody, bass and pad registers


# Chord token to three scale-degree indices. Stacking alternate degrees of a diatonic scale produces
# the correct triad quality for every root without a lookup table; token 7 is the tonic sus4.
def triad(c):
    return (0, 3, 4) if c == 7 else (c, c + 2, c + 4)


# Scale degree to MIDI note number: seven degrees per octave, twelve semitones per octave.
def deg_midi(d, root):
    return root + 12 * (d // 7) + SCALE[d % 7]


# Progression weights from voice leading rather than from taste. Two chords that share pitch classes
# move smoothly into each other, so shared-tone count is the base weight; a root moving by a fourth
# or a fifth is the strongest progression in tonal music, so it earns a bonus; and no chord may
# follow itself, because a bar line with nothing happening across it is not a progression.
def chord_matrix():
    pcs = [set(SCALE[d % 7] for d in triad(c)) for c in range(NCHORD)]
    W = np.zeros((NCHORD, NCHORD))
    for i in range(NCHORD):
        for j in range(NCHORD):
            if i != j:
                W[i, j] = 0.35 + len(pcs[i] & pcs[j]) + (0.6 if (j - i) % 7 in (3, 4) else 0.0)
    return W / W.sum(axis=1, keepdims=True)


print("vocabulary:", VOCAB, "tokens =", NPITCH, "degrees +", NCHORD, "chords + rest + tie")
print("chord 0 voiced from the pad root:", [deg_midi(d, PAD_ROOT) for d in triad(0)], "= A minor")
print("most likely chord after each of the eight:", [int(c) for c in chord_matrix().argmax(axis=1)])

## 2 · A corpus, computed rather than collected

The model has to learn from something, and on this platform that something must be derived in the
cells: no downloaded MIDI, no pasted score, nothing that a reader cannot regenerate. So we write a
small **grammar** and let it produce 1,200 eight-bar phrases — about 163,000 tokens, which is nothing
by language-model standards and a great deal for a model with twenty thousand parameters.

Rhythm comes from **recursive binary subdivision**. A bar starts as a single span of sixteen
sixteenths and splits in half with a probability that falls with depth: nearly always at the top,
about four times in five at the next level, half the time below that. Halves and quarters therefore
dominate, sixteenths are rare, and the result is metrically sane in a way that drawing durations from
a flat distribution never is. Each surviving leaf becomes an onset followed by ties, or, if it is off
the beat, a rest — which is how the phrasing gets its holes.

Pitch follows a **contour**: a direction of travel that persists and reverses about a third of the
time. On strong positions the melody lands on a tone of the current chord; between them it walks by
step through the scale. Two small rules keep it musical. Overshooting the top of the register flips
the direction, so the line cannot drift off the end of the instrument the way an unconstrained random
walk does; and a mild pull toward the middle of the range keeps the tessitura where a lead sounds
best.

The last ingredient matters most for how the finished track sounds. Each phrase is laid out as
**A A′ B A″** over two-bar cells: bars 3, 4, 7 and 8 replay a remembered cell with every onset pulled
onto the nearest tone of the new chord, so the shape survives the harmony changing underneath it.
That is the single thing this corpus is really teaching — that material *returns*. A model trained on
phrases that never repeat themselves will sample something that never repeats itself either, and a
piece of music with no returning hook is a texture, not a song.

It is worth being honest about what a grammar-generated corpus can and cannot give you. It cannot
teach the model anything the grammar does not already know: there is no borrowed chord in here, no
syncopation the subdivision rule cannot express, no phrase longer than eight bars. What it can do is
teach the *statistics* of those decisions — which chords tend to follow which, how long a note usually
lasts, where a phrase puts its rests, when a motif comes back — and a model that has absorbed those
statistics will make choices the grammar never made in any particular phrase.

Scale matters here more than it looks. A hundred and sixty thousand tokens against twenty thousand
parameters is about eight training tokens per parameter, and
that ratio is the whole reason the held-out curve in section 5 behaves: at eight tokens per parameter
there is no room to memorise, so the only way for the loss to fall is to generalise. An earlier draft
of this notebook used a tenth as much data, and the held-out loss turned upward after 150 updates while
the training loss kept dropping — textbook overfitting, invisible without the split.

In [ ]:
# Split probability by recursion depth. The first entry is 1.0 so a bar always divides at least once;
# the last is small so single sixteenths stay ornamental rather than becoming the default note value.
SPLIT_P = (1.0, 0.96, 0.82, 0.5, 0.22)


# One bar of durations by recursive binary subdivision. Because the probability of splitting falls
# with depth, quarters and eighths are common and single sixteenths are rare.
def rhythm(rng, span=SPB, depth=0):
    if span > 1 and rng.random() < SPLIT_P[min(depth, 4)]:
        return rhythm(rng, span // 2, depth + 1) + rhythm(rng, span // 2, depth + 1)
    return [span]


# One bar of grid tokens. Every rhythmic leaf becomes an onset plus ties, or a rest when it falls off
# the beat. Pitches land on chord tones at the strong positions and walk by step between them, along
# a contour that mostly persists; the register limits flip the direction rather than clamping it,
# which is what keeps the line moving instead of sticking to the top of its range.
def bar_tokens(rng, chord, prev, dirn, rest_p=0.12):
    toks, pos = [], 0
    tones = sorted({t + 7 * o for o in (0, 1, 2) for t in triad(chord) if 0 <= t + 7 * o < NPITCH})
    for dur in rhythm(rng):
        strong = pos % 4 == 0
        if not strong and rng.random() < rest_p:
            toks += [REST] * dur
            pos += dur
            continue
        if rng.random() < 0.34:
            dirn = -dirn                        # the contour changes its mind now and then
        reach = prev + dirn * ((2 if strong else 1) + int(rng.random() < 0.24))
        pool = tones if (strong or rng.random() < 0.45) else [prev + s for s in (-2, -1, 1, 2)]
        pool = [c for c in pool if 0 <= c < NPITCH and c != prev]
        if not pool:
            pool = [max(0, min(NPITCH - 1, prev + dirn))]
        d = min(pool, key=lambda c: abs(c - reach) + 0.1 * abs(c - 8))
        dirn = -1 if d > 15 else (1 if d < 2 else dirn)
        toks += [P0 + d] + [TIE] * (dur - 1)
        prev = d
        pos += dur
    return toks, prev, dirn


# Replay a remembered bar over a new chord: keep the rhythm exactly, pull every onset onto the
# nearest tone of the new triad. This is how a motif survives a chord change instead of clashing.
def reharmonise(bar, chord):
    tones = [t for t in (t + 7 * o for o in (0, 1, 2) for t in triad(chord)) if 0 <= t < NPITCH]
    return [x if x < P0 else P0 + min(tones, key=lambda c: abs(c - (x - P0))) for x in bar]


# 120 eight-bar phrases. Each opens on the tonic or the relative major, walks the chord matrix, and
# cadences somewhere that can turn around; bars 3, 4, 7 and 8 restate the first two-bar cell. Each
# bar is emitted as one chord token followed by its sixteen grid steps.
def make_corpus(n_phrases=1200, seed=SEED):
    rng = np.random.default_rng(seed)
    M, toks, onsets = chord_matrix(), [], []
    for _ in range(n_phrases):
        prog = [5 * int(rng.integers(0, 2))]
        while len(prog) < 8:
            prog.append(int(np.searchsorted(np.cumsum(M[prog[-1]]), rng.random())))
        if prog[-1] not in (0, 4, 6):
            prog[-1] = 0
        cell, prev, dirn = {}, 7, 1
        for b in range(8):
            if b in (2, 3, 6, 7):
                bar = reharmonise(cell[b % 2], prog[b])
            else:
                bar, prev, dirn = bar_tokens(rng, prog[b], prev, dirn)
                cell[b] = bar
            toks.append(C0 + prog[b])
            toks.extend(bar)
            onsets.append(sum(1 for x in bar if x >= P0))
    return np.array(toks, dtype=np.int64), onsets


DATA, ONSETS = make_corpus()
# A phrase is eight bars of one chord token plus sixteen grid steps, so exactly 136 tokens. Splitting
# on a multiple of 136 keeps whole phrases on one side or the other: a validation set that begins
# halfway through a phrase would share its opening cell with the training set, which is leakage.
PHRASE = 8 * (SPB + 1)
TRAIN, VALID = DATA[:1024 * PHRASE], DATA[1024 * PHRASE:]
print(f"corpus: {len(DATA)} tokens · {len(ONSETS)} bars · {len(set(DATA.tolist()))}/{VOCAB} tokens used")
print(f"training on {len(TRAIN)} tokens, holding out {len(VALID)} unseen tokens to validate against")
print(f"mean onsets per bar {np.mean(ONSETS):.2f} · ties are {100 * np.mean(DATA == TIE):.0f}% of the stream")
print("the first bar, as tokens:", DATA[:SPB + 1].tolist())

## 3 · The model: a GRU written out longhand

The network is a **gated recurrent unit** over that token stream, and every number in it is one of
19,975 trained parameters:

| tensor | shape | what it is |
|---|---|---|
| `emb` | 31 × 16 | a learned vector per token |
| `Wx` | 16 × 204 | the input side of all three gates at once |
| `Uzr` | 68 × 136 | the recurrent side of the update and reset gates |
| `Un` | 68 × 68 | the recurrent side of the candidate state |
| `b` | 204 | one bias per gate unit |
| `Wo`, `bo` | 68 × 31, 31 | the read-out to next-token logits |

The forward pass is four lines of arithmetic per step. The **update gate** `z` decides how much of the
previous hidden state to keep; the **reset gate** `r` decides how much of that state the candidate is
allowed to look at; the candidate `n` proposes a new state from the current token and the reset
history. The output is the two blended: `h = (1 − z) · n + z · h_prev`.

That one equation is the whole reason a gated network can write music at all. When `z` is near one the
state passes through untouched and the model is remembering — holding a motif across a bar line, or
keeping track of where it is in a phrase. When `z` is near zero the state is replaced and the model is
reacting to what just happened. Nothing tells it which to do where; the gates are trained, and what
they learn is the corpus's phrase structure.

Why 68 hidden units and 16 embedding dimensions? Because that combination lands the parameter count
at 19,975, and the platform's `weights` axis is calibrated with its ideal at 20,000 — large enough
that the model has clearly learned something, small enough that a reader can hold the whole thing in
their head and check it by hand.

At that size the model has room for a handful of things and no more. Sixty-eight units of state can
carry where it is in the bar, whether a note is sounding, roughly where the melody sits in its
register, and a compressed impression of the last phrase. They cannot carry a plan for the next
thirty-two bars. That limit is the reason the eight-section arc in section 8 is written down by a
human rather than sampled: asking this network to invent long-range form would be asking it for
something its state has no room to hold, and the honest move is to say so rather than to pretend.

In [ ]:
EMB, HID = 16, 68


# Scaled Gaussian initialisation with the biases at zero: each matrix is divided by the square root
# of its input width so the pre-activations start near unit variance and the gates start unsaturated.
# The generator is seeded, which makes the starting point part of the reproducible record.
def init_params(seed=SEED):
    rng = np.random.default_rng(seed + 1)
    def g(rows, cols, s=None):
        return rng.standard_normal((rows, cols)) * (s or 1.0 / math.sqrt(rows))
    return {"emb": g(VOCAB, EMB, 0.35), "Wx": g(EMB, 3 * HID), "Uzr": g(HID, 2 * HID),
            "Un": g(HID, HID), "b": np.zeros(3 * HID), "Wo": g(HID, VOCAB), "bo": np.zeros(VOCAB)}


# One recurrent step, batched over rows. The three gate pre-activations share one input matmul; the
# update and reset gates share one recurrent matmul; the candidate needs its own because it reads the
# reset-gated state rather than the raw one. Everything the backward pass will need comes back too.
def gru_step(P, e, h):
    g = e @ P["Wx"] + P["b"]
    zr = 1.0 / (1.0 + np.exp(-(g[:, :2 * HID] + h @ P["Uzr"])))
    z, r = zr[:, :HID], zr[:, HID:]                 # update gate, reset gate
    rh = r * h
    n = np.tanh(g[:, 2 * HID:] + rh @ P["Un"])      # the candidate state
    hn = (1.0 - z) * n + z * h                      # keep or replace, per unit, learned
    return hn, (e, h, z, r, rh, n, hn)


# Run a batch of token sequences through the network from a zero state, keeping one cache per step.
def forward(P, X):
    B, T = X.shape
    h, cache, logits = np.zeros((B, HID)), [], np.empty((B, T, VOCAB))
    for t in range(T):
        h, c = gru_step(P, P["emb"][X[:, t]], h)
        logits[:, t] = h @ P["Wo"] + P["bo"]
        cache.append(c)
    return logits, cache


PARAMS = init_params()
print("parameters:", {k: v.shape for k, v in PARAMS.items()})
print("total:", sum(v.size for v in PARAMS.values()), "· ideal for the weights axis: 20,000")

## 4 · The backward pass, and proof that it is right

Backpropagation through time walks the cache in reverse carrying one gradient with respect to the
hidden state. At each step that gradient splits three ways: into the candidate, scaled by `1 − z`;
into the update gate, scaled by the difference `h_prev − n`, because that is what `z` actually
chooses between; and straight through the `z · h_prev` shortcut, which is the path that lets gradient
reach back across dozens of steps without vanishing.

Then the candidate's share has to get back past the reset gate, and this is the only term in a GRU
that is easy to get wrong. The candidate sees `r * h_prev`, so the gradient arriving there reaches
`h_prev` **twice** — once directly, scaled by `r`, and once through `r` itself, since `r` was computed
from `h_prev`. Miss the second route and training still runs, the loss still falls, and the model is
quietly worse than it should be. Nothing warns you.

So we check it. A **finite-difference check** nudges twenty-eight individual parameters by ±10⁻⁵,
measures how the loss actually moves, and compares that with the analytic gradient. Agreement to
about one part in 10⁸ is what a correct backward pass looks like in float64. A missing term shows up
immediately as a relative error around 10⁻¹ on the tensors that term feeds. It costs a few hundred
extra forward passes and it is the difference between believing the gradients and knowing them.

In [ ]:
# Backpropagation through time. `dhp` accumulates the three routes into the previous hidden state:
# the shortcut through the update gate, the reset-gated path into the candidate, and the reset gate
# itself. `np.add.at` scatters the embedding gradient because a token can appear many times in a
# batch and its rows must accumulate rather than overwrite.
def backward(P, X, cache, dlogits):
    G = {k: np.zeros_like(v) for k, v in P.items()}
    dh = np.zeros((X.shape[0], HID))
    for t in range(X.shape[1] - 1, -1, -1):
        e, hp, z, r, rh, n, hn = cache[t]
        dl = dlogits[:, t]
        G["Wo"] += hn.T @ dl
        G["bo"] += dl.sum(axis=0)
        dh = dh + dl @ P["Wo"].T
        dz, dn, dhp = dh * (hp - n), dh * (1.0 - z), dh * z
        an = dn * (1.0 - n * n)                     # through the tanh of the candidate
        G["Un"] += rh.T @ an
        drh = an @ P["Un"].T
        dhp += drh * r                              # the direct route back into h_prev
        azr = np.concatenate([dz * z * (1.0 - z), (drh * hp) * r * (1.0 - r)], axis=1)
        G["Uzr"] += hp.T @ azr
        dhp += azr @ P["Uzr"].T                     # and the route through both gates
        dg = np.concatenate([azr, an], axis=1)
        G["Wx"] += e.T @ dg
        G["b"] += dg.sum(axis=0)
        np.add.at(G["emb"], X[:, t], dg @ P["Wx"].T)
        dh = dhp
    return G


# Mean cross-entropy of predicting every token from the one before it, and its gradient. Subtracting
# the row maximum before the exponential is the standard guard against overflow; subtracting one at
# the target index turns the softmax into the softmax-cross-entropy gradient in a single line.
def loss_and_grad(P, batch):
    X, Y = batch[:, :-1], batch[:, 1:]
    logits, cache = forward(P, X)
    p = np.exp(logits - logits.max(axis=2, keepdims=True))
    p /= p.sum(axis=2, keepdims=True)
    B, T = Y.shape
    bi, ti = np.arange(B)[:, None], np.arange(T)[None, :]
    loss = float(-np.log(np.maximum(p[bi, ti, Y], 1e-12)).mean())
    p[bi, ti, Y] -= 1.0
    return loss, backward(P, X, cache, p / (B * T))


# The gradient check: central differences on four random coordinates of each of the seven tensors.
# Coordinates whose true gradient is near zero are skipped, because a relative error against a
# gradient of 1e-9 measures float noise rather than correctness.
check = np.stack([DATA[i:i + 9] for i in (3, 900)])
probe = {k: v.copy() for k, v in PARAMS.items()}     # nudge a COPY: adding and subtracting 1e-5 does
base, grads = loss_and_grad(probe, check)            # not restore a float exactly, and the weights
rng, worst = np.random.default_rng(SEED + 3), 0.0    # training starts from must stay pristine
for name in probe:
    for _ in range(4):
        idx = tuple(int(rng.integers(0, s)) for s in probe[name].shape)
        probe[name][idx] += 1e-5
        up = loss_and_grad(probe, check)[0]
        probe[name][idx] -= 2e-5
        down = loss_and_grad(probe, check)[0]
        probe[name][idx] += 1e-5
        numeric = (up - down) / 2e-5
        if abs(numeric) > 1e-7:
            worst = max(worst, abs(numeric - grads[name][idx]) / abs(numeric))
print(f"loss on the check batch {base:.4f} · uniform guessing would score {math.log(VOCAB):.4f}")
print(f"worst relative gradient error over 28 coordinates: {worst:.2e}")

## 5 · Training

Adam, 1,200 updates, each on 64 windows of 48 consecutive tokens drawn from anywhere in the training
phrases. Forty-eight tokens is just under three bars, which is deliberate: the two-bar cell and its
restatement both fall inside one window, so the returning-motif structure is something the model can
see rather than something it has to infer across a truncation boundary.

A hundred and seventy-six of the 1,200 phrases are **held out** and never trained on, and every 150
updates the same fixed batch of unseen windows is scored. That second curve is the only thing that distinguishes a model
which has learned the idiom from one which has memorised the corpus, and it costs almost nothing to
compute. If the held-out loss tracks the training loss down and stays beside it, the model has
generalised. If it flattens while the training loss keeps falling, training should stop — and printing
both numbers means nobody has to take that on trust.

Three details earn their place. Gradients are clipped to a global norm of 5, because a recurrent
network occasionally produces one enormous gradient and a single unclipped step can undo fifty good
ones. The learning rate drops to a third for the last third of training, which is the cheapest
version of a schedule and worth about a tenth of a nat here. And Adam's bias correction is written
out rather than skipped, since at step 1 the raw moment estimates are ten times too small.

Watch both numbers. Cross-entropy starts near `ln 31 ≈ 3.43`, which is exactly what a model that has
learned nothing scores, and settles near 1.4 nats per token on the training batches and near 1.45 on
phrases it has never seen. That is not a rounding detail. 3.43 nats means all 31 tokens are equally
plausible at every step; 1.45 nats is a perplexity of about four, meaning the model has narrowed 31
possibilities down to roughly four live candidates. It has learned that ties follow onsets, that chord
tokens arrive once a bar, that phrases restate their opening cell, and that a line which went up tends
to come back down.

The gap between the two curves is the number that matters, and here it is a few hundredths of a nat.
A model that had memorised its corpus would show a training loss far below its held-out loss, and it
would sample material that is recognisably a phrase from the training set rather than something new in
the same idiom.

In [ ]:
# Adam with global-norm gradient clipping and a step-down at two thirds of training. Returns both loss
# histories so the fall can be plotted at the end rather than merely asserted in prose. The validation
# batch is fixed and evenly spaced through the held-out phrases, so its curve is comparable across
# updates; it is scored with the same function as the training batch and its gradient is discarded.
def train(P, data, valid, updates=1200, T=48, B=64, lr=0.02, seed=SEED):
    rng = np.random.default_rng(seed + 2)
    vstart = np.arange(0, len(valid) - T - 1, (len(valid) - T - 1) // 32)[:32]
    vbatch = np.stack([valid[i:i + T + 1] for i in vstart])
    m = {k: np.zeros_like(v) for k, v in P.items()}
    v = {k: np.zeros_like(v) for k, v in P.items()}
    history, checks = [], []
    for step in range(1, updates + 1):
        # Windows start anywhere, so a batch mixes phrase openings, middles and cadences; the extra
        # token on the end is the last target, since every window predicts itself shifted by one.
        start = rng.integers(0, len(data) - T - 1, size=B)
        loss, G = loss_and_grad(P, np.stack([data[i:i + T + 1] for i in start]))
        # One global norm across all seven tensors, not one per tensor: clipping them separately would
        # change the gradient's direction, and only its length is the problem.
        norm = math.sqrt(sum(float((g * g).sum()) for g in G.values()))
        clip = min(1.0, 5.0 / max(norm, 1e-9))
        rate = lr * (0.35 if step > updates * 2 // 3 else 1.0)
        for k in P:
            m[k] = 0.9 * m[k] + 0.1 * G[k] * clip
            v[k] = 0.999 * v[k] + 0.001 * (G[k] * clip) ** 2
            # Adam's bias correction, written out: at step 1 the raw moment estimates are ten and a
            # thousand times too small, and skipping this makes the first dozen updates far too timid.
            mhat, vhat = m[k] / (1 - 0.9 ** step), v[k] / (1 - 0.999 ** step)
            P[k] -= rate * mhat / (np.sqrt(vhat) + 1e-8)
        history.append(loss)
        if step % 150 == 0 or step == 1:
            checks.append((step, loss_and_grad(P, vbatch)[0]))
            print(f"  update {step:>4}  train {loss:.4f}   held-out {checks[-1][1]:.4f}")
    return history, checks


HISTORY, CHECKS = train(PARAMS, TRAIN, VALID)
# The weights ship as float32, so the render has to use float32 values or the published file would
# reproduce something subtly different. Rounding here, once, makes the two paths bit-identical.
MODEL = {k: v.astype(np.float32).astype(np.float64) for k, v in PARAMS.items()}
print(f"training cross-entropy {HISTORY[0]:.3f} -> {np.mean(HISTORY[-20:]):.3f} nats per token")
gap = CHECKS[-1][1] - np.mean(HISTORY[-20:])
print(f"held-out cross-entropy {CHECKS[0][1]:.3f} -> {CHECKS[-1][1]:.3f}, a gap of {gap:+.3f} nats: the "
      f"two curves land together, so this is a learned idiom and not a memorised corpus")
print(f"perplexity on unseen phrases {math.exp(CHECKS[-1][1]):.2f} — about four live choices out of {VOCAB}")
print("weights rounded to the float32 precision they ship in, so the re-render can match exactly")

## 6 · Sampling: three voices out of one model

Now the model writes. Sampling is **constrained**: at the first position of a bar every logit except
the eight chord tokens is set to minus infinity, and at the sixteen grid positions the chord tokens
are masked out along with any degree outside the register this voice is allowed to use. The model
still chooses — it simply cannot choose something unplayable. This is the same trick a compiler-aware
code model uses to stay syntactically valid, and it is worth being precise about what it does and does
not do: it removes options, it never adds a note. If a bar comes back with fewer than two onsets it is
redrawn from the state it started in, up to four times, which keeps an unlucky draw from leaving a
hole in the arrangement.

Three passes, three temperatures, three registers, one set of weights:

- **HOOK** — eight bars, temperature 0.85, the middle of the range. This is the tune.
- **RIFF** — eight bars, temperature 0.70, the bottom octave. It becomes the bass line, and its
  onsets become the kick pattern.
- **LIFT** — eight bars, temperature 1.05, the top. It becomes the break, and the counter-melody over
  the final drop.

Temperature is the one knob worth understanding here. Dividing the logits by a number below one
sharpens the distribution toward whatever the model is confident about; above one it flattens the
distribution and lets unlikely tokens through. A hook wants confidence, because a hook has to be
memorable. A break wants surprise, because its job is to make the return feel like a return.

The redraw rule deserves naming too, because it is a real intervention and not a formality: it is
rejection sampling, and it biases the output distribution toward bars with at least two onsets. The
bias is small — at these temperatures most bars pass first time — but it is not nothing, and pretending
otherwise would be the kind of quiet dishonesty that makes generated music impossible to evaluate.
What it buys is an arrangement with no accidental silence in the middle of a drop. What it costs is
that the very sparsest thing the model could have said never gets said.

In [ ]:
# Constrained decoding. `chord_only` and `grid_only` are additive log-domain masks: minus a billion
# is numerically minus infinity after the softmax, so masked tokens cannot be drawn at all. Sampling
# itself is an inverse-CDF lookup on the cumulative distribution, which is deterministic given the
# generator. A bar with fewer than two onsets is redrawn from the hidden state it started from.
def sample_section(P, rng, bars, temp, lo, hi):
    chord_only, grid_only = np.full(VOCAB, -1e9), np.full(VOCAB, -1e9)
    chord_only[C0:] = 0.0
    grid_only[[REST, TIE]] = 0.0
    grid_only[P0 + lo:P0 + hi + 1] = 0.0
    # The state persists across bars, so bar 5 is written in the light of bars 1 to 4; only the redraw
    # rewinds it. Sampling starts from a zero state and a rest, which is the model's idea of silence.
    h, tok, chords, grid = np.zeros((1, HID)), REST, [], []
    for _ in range(bars):
        for attempt in range(4):
            h0, tok0, row, chord = h.copy(), tok, [], None
            for pos in range(SPB + 1):
                h, _ = gru_step(P, P["emb"][[tok]], h)
                z = (h @ P["Wo"] + P["bo"])[0] / temp + (chord_only if pos == 0 else grid_only)
                p = np.exp(z - z.max())
                p /= p.sum()
                tok = int(min(VOCAB - 1, np.searchsorted(np.cumsum(p), rng.random())))
                if pos == 0:
                    chord = tok - C0
                else:
                    row.append(tok)
            if sum(1 for x in row if x >= P0) >= 2 or attempt == 3:
                break
            h, tok = h0, tok0
        chords.append(chord)
        grid.append(row)
    return chords, np.array(grid, dtype=np.int64)


# Grid tokens to playable notes: (step, length in steps, MIDI note). A tie extends whatever is
# sounding; a tie with nothing sounding is silence, so no token order can conjure a note out of
# nowhere. `hold` shortens every note slightly to leave audible space between repeated pitches.
def notes_from(grid, root, hold=1.0, tr=0):
    out, cur = [], None
    for i, t in enumerate(grid.reshape(-1)):
        if t >= P0:
            out += [cur] if cur else []
            cur = [i, 1, int(t - P0)]
        elif t == TIE and cur:
            cur[1] += 1
        else:
            out += [cur] if cur else []
            cur = None
    out += [cur] if cur else []
    return [(s, max(1, int(round(L * hold))), deg_midi(d + tr, root)) for s, L, d in out]

Here is the material the model actually produced. Read each grid as a piano roll, one row per bar and
one column per sixteenth: `0` is a rest, `1` holds the previous note, and anything from 2 upward is an
onset on that degree. The chord list printed above each grid is the model's own harmony for those
eight bars — it was never handed a progression to follow.

In [ ]:
# Sample the three voices from one generator used in a fixed order, so the composition is a pure
# function of the weights. That is exactly what makes the re-render in section 12 possible: the same
# weights and the same seed have to produce the same tokens before they can produce the same audio.
def compose(P, seed=1109):
    rng = np.random.default_rng(seed)
    return {"HOOK": sample_section(P, rng, 8, 0.85, 3, 13),
            "RIFF": sample_section(P, rng, 8, 0.70, 0, 7),
            "LIFT": sample_section(P, rng, 8, 1.05, 6, 17)}


VOICES = compose(MODEL)
for name, (chords, grid) in VOICES.items():
    print(f"{name}  chords {chords}  onsets per bar {[int((r >= P0).sum()) for r in grid]}")
for row in VOICES["HOOK"][1][:4]:
    print("   ", " ".join(f"{t:2d}" for t in row))
print("HOOK bar 1 as notes (step, length, MIDI):", notes_from(VOICES["HOOK"][1][:1], MEL_ROOT)[:6])

## 7 · Instruments, from first principles

No sample libraries and no oscillator classes: a note here is a sum of sine waves times an envelope.
Every pitched instrument is **additive**, which means the timbre is literally a list of numbers — pick
an amplitude for each harmonic and you have chosen a sound. Amplitudes falling off as 1/n approximate
a sawtooth and sound bright and synthetic, which is what a lead wants. Three copies of that, detuned
by a few thousandths of a semitone, beat slowly against each other and turn one thin voice into a wide
one; it is the cheapest convincing trick in synthesis. The choir's amplitudes bulge at the fourth and
fifth harmonic, roughly where a vowel's formant sits, and that bulge is the whole difference between
"pad" and "voices".

Percussion is noise and decay. A kick is a sine whose frequency sweeps from 164 Hz down to 44 Hz
inside a tenth of a second, which is why the phase has to come from a cumulative sum of the frequency
rather than from a constant times `t`. A hat is white noise differenced twice under a very fast
exponential decay — differencing is a first-order high-pass filter, so two of them tilt the spectrum
upward by 12 dB per octave and turn flat noise into something bright and metallic. A snare stacks the
two ideas. The riser under each build is noise whose amplitude grows as `t²`, and the impact at each
drop is a 46 Hz thump with a noise transient riding on it.

Why additive rather than the subtractive synthesis every hardware synthesiser uses — a rich waveform
through a resonant filter? Because a resonant filter is a recursive difference equation, and a
recursive equation over seven and a half million samples in interpreted Python is minutes of work per
pass. Additive synthesis inverts the problem: instead of removing what you do not want from a bright
source, you add only what you do want, one vectorised sine at a time, and the spectrum is exactly the
list of numbers you wrote down. In an environment with no scipy that is not a compromise; it is the
approach with the fewest moving parts.

The one filter needed repeatedly is a **boxcar moving average**, computed off a cumulative sum so it
costs a single pass over the signal no matter how wide it is. Subtracting a smoothed copy of a signal
from itself leaves its high band, so `x + g·(x − smooth(x))` is a broad high shelf: an air control for
four array operations and no scipy. Every filter in this notebook is one of those two ideas.

In [ ]:
LEAD_H = (1.0, .62, .45, .34, .26, .2, .16, .12, .09, .07, .055, .04)      # saw-ish: amplitude ~ 1/n
PAD_H = (1.0, .5, .3, .14, .07, .04)                                      # soft, few harmonics
CHOIR_H = (1.0, .78, .5, .42, .5, .34, .2, .13, .09)                      # a formant bulge at 4 and 5
BASS_H = (1.0, .55, .3, .13, .06)
PLUCK_H = (1.0, .7, .5, .38, .28, .2, .14, .1, .07)
SHIM_H = (1.0, .6, .42, .3, .22, .16, .11)


def hz(midi):
    return 440.0 * 2.0 ** ((midi - 69) / 12.0)


# Attack, decay to a sustain level, then an exponential release, all four given in seconds. The
# attack curve is raised to 1.6 so it starts gently: a linear attack on a bright sound clicks.
def envelope(L, attack, decay, sustain, release):
    e = np.ones(L)
    na = min(int(attack * SR), L)
    nd = min(int(decay * SR), max(0, L - na))
    nr = min(int(release * SR), L)
    e[:na] = np.linspace(0.0, 1.0, na) ** 1.6
    e[na:na + nd] = np.linspace(1.0, sustain, nd)
    e[na + nd:] = sustain
    e[L - nr:] *= np.exp(-np.linspace(0.0, 5.0, nr))
    return e


# Additive synthesis: one sine per harmonic, optionally in three detuned copies, optionally with
# vibrato. Harmonics past the Nyquist limit are dropped rather than allowed to alias down into the
# audible range as a wrong note, which is the failure mode of naive digital sawtooths.
def tone(f, L, harmonics, detune=0.0, vib=0.0, vibf=5.2):
    # Vibrato is divided by the fundamental so that its depth stays constant in semitones rather than
    # in hertz: the same setting then means the same wobble on a bass note and on a lead two octaves up.
    t = np.arange(L) / SR
    phase = 2.0 * np.pi * t
    if vib:
        phase = phase + vib * np.sin(2.0 * np.pi * vibf * t) / max(f, 1.0)
    y = np.zeros(L)
    for df in ((0.0,) if detune == 0.0 else (-detune, 0.0, detune)):
        for k, a in enumerate(harmonics, start=1):
            if f * (1.0 + df) * k > 0.44 * SR:
                break
            y += a * np.sin(f * (1.0 + df) * k * phase)
    return y / (1.0 if detune == 0.0 else 3.0)


# Boxcar moving average, edge-padded to keep the length, taken off a cumsum so width costs nothing.
def smooth(x, w):
    xp = np.concatenate([np.full(w // 2, x[0]), x, np.full(w - w // 2 - 1, x[-1])])
    c = np.concatenate([[0.0], np.cumsum(xp)])
    return (c[w:] - c[:-w]) / w


# Differencing is a first-order high-pass; n passes tilt the spectrum by 6n dB per octave.
def hp(x, n=1):
    for _ in range(n):
        x = np.concatenate([[0.0], np.diff(x)])
    return x


# A kick: the frequency sweep means the phase must be the cumulative sum of frequency, not f times t.
def kick(L):
    t = np.arange(L) / SR
    return np.sin(2.0 * np.pi * np.cumsum(44.0 + 120.0 * np.exp(-t * 34.0)) / SR) * np.exp(-t * 7.0)


# A snare is the two ideas stacked: a short tuned body under a burst of high-passed noise. The noise
# offset differs per grid position, so no two hits in a bar are the same sample of noise twice.
def snare(L, noise, off):
    t = np.arange(L) / SR
    return (np.sin(2.0 * np.pi * 188.0 * t) * np.exp(-t * 26.0) * 0.45
            + hp(noise[off:off + L]) * np.exp(-t * 19.0) * 0.7)


def hat(L, noise, off):
    return hp(noise[off:off + L], 2) * np.exp(-np.arange(L) / SR * 105.0) * 0.5


def riser(L, noise):
    t = np.arange(L) / SR
    return hp(np.resize(noise, L)) * (t / t[-1]) ** 2.4 * 0.8


def impact(L, noise):
    t = np.arange(L) / SR
    return np.sin(2.0 * np.pi * 46.0 * t) * np.exp(-t * 3.4) + hp(noise[:L]) * np.exp(-t * 8.0) * 0.35


print("a second of lead at concert A:", tone(440.0, SR, LEAD_H, detune=0.0035).nbytes, "bytes of float64")

## 8 · The arrangement

The form is the one place where a human decision belongs, so it is written down honestly as a table
rather than hidden in a loop: eight sections, and for each one a level for every instrument, which
sampled material the lead plays, whether a riser or an impact marks its entrance, and how far the
material is transposed.

Two columns do most of the emotional work. **Intensity** is the section's place in the arc — the
statement and the resolution sit about five decibels under the drops, which is roughly the loudness
range of the epic tracks this piece is modelled on, and is the number section 10 will measure. Set it
too wide and the quiet parts vanish on a phone; too narrow and there is no arc left to hear.
**Transposition** shifts a section by whole scale degrees: the return of the hook comes back a
diatonic third higher, the oldest device in the genre for making a last chorus feel larger, and the
second build moves too so that it does not simply echo the first. Because the alphabet is diatonic,
transposing by two degrees maps chord tones onto chord tones — the tune goes up, and it stays in key.

Everything else is derived, and it is worth spelling out how, because "derived from the model" is a
claim that deserves detail. Pad and choir voice whichever chord the model chose for that bar, and the
bass plays RIFF two octaves down.

The kick fires on RIFF's onsets that land on eighths, so the drums lock to the model's own rhythm
rather than to a metronome; only the downbeat is guaranteed. Hats fill the steps where the hook is
*not* playing, so the two interlock instead of colliding. The pluck picks up
chord tones in the hook's rests, and the shimmer doubles the hook two octaves up, which is where most
of the brightness in the finished mix comes from. So the percussion has no pattern of its own: change
the weights and the drums change with them, which is the opposite of a drum loop with a model bolted
on beside it.

In [ ]:
# start, stop, name, material (lead first, then any counter-melodies), intensity, then a level for
# lead, pad, choir, bass, pluck, kick, snare and hat, then the entrance effect and the transposition
# in scale degrees. This table is the arrangement: reorder it, and the piece is a different piece.
FORM = (
    (0, 8, "Statement", ("HOOK",), 1.33, 0.50, 0.66, 0.44, 0.00, 0.34, 0.00, 0.00, 0.18, "", 0),
    (8, 16, "Verse", ("HOOK",), 1.00, 0.60, 0.56, 0.30, 0.66, 0.38, 0.60, 0.00, 0.26, "", 0),
    (16, 24, "Build", ("HOOK",), 1.05, 0.66, 0.50, 0.36, 0.74, 0.46, 0.66, 0.46, 0.38, "riser", 0),
    (24, 32, "Drop", ("HOOK",), 1.00, 0.95, 0.62, 0.56, 1.00, 0.48, 1.00, 0.85, 0.52, "impact", 0),
    (32, 40, "Break", ("LIFT",), 1.08, 0.56, 0.58, 0.44, 0.34, 0.38, 0.34, 0.00, 0.22, "", 0),
    (40, 48, "Build II", ("LIFT",), 1.00, 0.70, 0.54, 0.46, 0.80, 0.48, 0.72, 0.58, 0.42, "riser", 3),
    (48, 56, "Return", ("HOOK", "LIFT"), 1.00, 1.00, 0.64, 0.60, 1.00, 0.50, 1.00, 0.90, 0.55, "impact", 2),
    (56, 64, "Resolution", ("HOOK",), 1.40, 0.58, 0.60, 0.46, 0.38, 0.34, 0.34, 0.00, 0.18, "", 0),
)
# The counter-melody level, how much shimmer rides on the pluck level, where a build starts relative
# to where it ends, and how far the resolution fades across its eight bars.
COUNTER_G, SHIM_MUL, BUILD_LO, RES_FADE = 0.30, 0.18, 0.55, 0.30
for s0, s1, name, keys, inten, *rest in FORM:
    print(f"  bars {s0:>2}-{s1 - 1:<2} {name:<11} {'+'.join(keys):<10} intensity {inten:.2f}  {rest[8] or '-'}")

## 9 · Rendering

Seven buses — lead, pad, choir, bass, pluck, drums and effects — each a mono float64 array as long as
the piece plus two seconds of room for the reverb to decay into. Notes are stamped into them one at a
time.

The one optimisation worth its complexity is a **memo**. Every instrument is a pure function of pitch
and length, so a note is synthesised once and reused everywhere it recurs. Since the arrangement
repeats material by design — that is what a hook is — most of the 64 bars cost almost nothing after
the first pass, and the whole 170-second render takes a few seconds instead of a few minutes. Caching
also guarantees that two occurrences of the same note are bit-identical, which is a small gift to the
reproducibility check later.

Then the mix. Each bus is panned with a constant-power law, cosine to the left and sine to the right,
so moving a sound across the image does not change how loud it is. The wide buses get a few
milliseconds of delay on the right channel instead: below about 30 ms the ear reads a delay as
direction and space rather than as an echo, which is the Haas effect, and it opens up the stereo
picture without any filtering. One reverb send feeds twelve delay taps between 37 and 487 ms,
alternating sides and darkened by a moving average, and that is enough to put the whole arrangement in
one plausible room.

In [ ]:
# Add a signal into a bus at a sample offset, clipped to the bus length so a long tail near the end
# of the piece is truncated instead of raising.
def place(bus, sig, at, g=1.0):
    L = min(len(sig), len(bus) - at)
    if L > 0 and at >= 0:
        bus[at:at + L] += g * sig[:L]


# Stamp every section of the form into the seven buses. Pitches come from the model's tokens; the
# levels, the order of the sections and the transpositions come from FORM. `voice` is the memo: an
# instrument is a pure function of pitch and length, so each distinct note is synthesised once.
def render(voices, seed=4211):
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(3 * SR)
    bus = {k: np.zeros(NS + TAIL) for k in ("lead", "pad", "choir", "bass", "pluck", "drum", "fx")}
    memo = {}
    def voice(key, make):
        if key not in memo:
            memo[key] = make()
        return memo[key]
    riff = voices["RIFF"][1]
    for s0, s1, name, keys, inten, gl, gp, gc, gb, gk, gki, gsn, ght, fx, tr in FORM:
        chords, grid = voices[keys[0]]
        gl, gp, gc, gb, gk, gki, gsn, ght = (inten * v for v in (gl, gp, gc, gb, gk, gki, gsn, ght))
        for b in range(s0, s1):
            i, at = b - s0, b * BAR
            # builds rise across their eight bars, the resolution falls, everything else sits still
            ramp = (BUILD_LO + (1.0 - BUILD_LO) * i / 7.0 if "Build" in name else
                    1.0 - RES_FADE * i / 7.0 if name == "Resolution" else 1.0)
            ch = chords[i] if chords[i] == 7 else (chords[i] + tr) % 7
            tones, pl = list(triad(ch)), BAR + 6 * STEP
            place(bus["pad"], voice(("pad", ch), lambda: sum(
                tone(hz(deg_midi(d, PAD_ROOT)), pl, PAD_H, 0.004, 0.9) for d in tones)
                * envelope(pl, 0.35, 0.5, 0.75, 0.55)), at, gp * ramp * 0.32)
            place(bus["choir"], voice(("choir", ch), lambda: sum(
                tone(hz(deg_midi(d, PAD_ROOT + 12)), BAR, CHOIR_H, 0.007, 1.6) for d in tones)
                * envelope(BAR, 0.5, 0.4, 0.8, 0.5)), at, gc * ramp * 0.24)
            for st, ln, mid in notes_from(grid[i:i + 1], MEL_ROOT, 0.98, tr):
                nl = ln * STEP
                place(bus["lead"], voice(("lead", mid, nl), lambda: tone(hz(mid), nl, LEAD_H, 0.0035, 2.4)
                      * envelope(nl, 0.012, 0.09, 0.72, 0.1)), at + st * STEP, gl * ramp * 0.5)
                if gl > 0.9:                        # the drops double the tune an octave up
                    place(bus["lead"], voice(("oct", mid, nl), lambda: tone(hz(mid + 12), nl, LEAD_H[:6], 0.004)
                          * envelope(nl, 0.02, 0.1, 0.6, 0.12)), at + st * STEP, gl * 0.16)
            for extra in keys[1:]:                  # a counter-melody, so the last drop is not the first
                for st, ln, mid in notes_from(voices[extra][1][i:i + 1], MEL_ROOT + 12, 0.9, tr):
                    nl = ln * STEP
                    place(bus["pluck"], voice(("counter", mid, nl), lambda: tone(hz(mid), nl, PLUCK_H, 0.005)
                          * envelope(nl, 0.02, 0.2, 0.4, 0.1)), at + st * STEP, COUNTER_G * gl * ramp)
            if gb > 0:                              # the bass is RIFF, two octaves down
                for st, ln, mid in notes_from(riff[i:i + 1], BASS_ROOT, 0.9, tr):
                    nl = ln * STEP
                    place(bus["bass"], voice(("bass", mid, nl), lambda: tone(hz(mid), nl, BASS_H)
                          * envelope(nl, 0.006, 0.06, 0.8, 0.08)), at + st * STEP, gb * ramp * 0.62)
            for st in range(SPB):
                if grid[i][st] == REST and st % 2 == 0:      # pluck the chord where the hook rests
                    mid = deg_midi(tones[(st // 2) % 3] + 7, MEL_ROOT)
                    place(bus["pluck"], voice(("pluck", mid), lambda: tone(hz(mid), 2 * STEP, PLUCK_H)
                          * envelope(2 * STEP, 0.004, 0.25, 0.05, 0.05)), at + st * STEP, gk * ramp * 0.3)
                if grid[i][st] >= P0:                       # and shimmer two octaves above it
                    mid = deg_midi(int(grid[i][st] - P0) + tr, MEL_ROOT + 24)
                    place(bus["pluck"], voice(("shimmer", mid), lambda: tone(hz(mid), 3 * STEP, SHIM_H)
                          * envelope(3 * STEP, 0.003, 0.3, 0.06, 0.06)), at + st * STEP, SHIM_MUL * gk * ramp)
                if gki > 0 and (st == 0 or (riff[i][st] >= P0 and st % 2 == 0) or (gki > 0.9 and st == 8)):
                    place(bus["drum"], voice(("kick",), lambda: kick(int(0.42 * SR))), at + st * STEP, gki * ramp * 0.85)
                if gsn > 0 and (st in (4, 12) or (grid[i][st] >= P0 and st % 4 == 2 and i % 4 == 3)):
                    place(bus["drum"], voice(("snare", st), lambda: snare(int(0.22 * SR), noise, st * 977)),
                          at + st * STEP, gsn * ramp * 0.42)
                if ght > 0 and (grid[i][st] < P0 or st % 2 == 1):
                    place(bus["drum"], voice(("hat", st), lambda: hat(int(0.06 * SR), noise, st * 1381)),
                          at + st * STEP, ght * ramp * (0.5 if st % 4 else 0.85))
        if fx == "riser":
            place(bus["fx"], riser((s1 - s0) * BAR, noise), s0 * BAR, 0.16)
        elif fx == "impact":
            place(bus["fx"], voice(("impact",), lambda: impact(int(2.2 * SR), noise)), s0 * BAR, 0.55)
    return bus


# Where each bus sits across the image, how many samples of Haas delay widens it, how much of it goes
# to the reverb, where the twelve taps land in seconds, and how much air the shelf adds. Bass and drums
# stay dead centre because low frequencies carry no directional information the ear can use anyway.
PAN = {"lead": 0.47, "pad": 0.5, "choir": 0.5, "bass": 0.5, "pluck": 0.62, "drum": 0.5, "fx": 0.44}
WIDE = {"lead": 0, "pad": 380, "choir": 530, "bass": 0, "pluck": 190, "drum": 0, "fx": 640}
SEND = {"lead": 0.28, "pad": 0.42, "choir": 0.5, "bass": 0.05, "pluck": 0.34, "drum": 0.1, "fx": 0.3}
TAPS = (0.037, 0.061, 0.089, 0.113, 0.149, 0.181, 0.223, 0.271, 0.317, 0.367, 0.421, 0.487)
AIR_G = 0.55


# Constant-power panning, a few milliseconds of Haas delay on the wide buses, one reverb send into
# twelve darkened taps that alternate sides, and a broad air shelf across both channels.
def mix(bus):
    L = len(bus["lead"])
    st, send = np.zeros((L, 2)), np.zeros(L)
    for k, sig in bus.items():
        p, d = PAN[k] * math.pi / 2.0, WIDE[k]
        st[:, 0] += math.cos(p) * sig
        st[d:, 1] += math.sin(p) * (sig if d == 0 else sig[:L - d])
        send += SEND[k] * sig
    # Darkening the send before the taps is what makes it read as a room rather than as twelve echoes:
    # real reflections lose their high end to the air and the walls on every bounce.
    dark = smooth(send, 5)
    for i, delay in enumerate(TAPS):
        n = int(delay * SR)
        g = 0.42 * 0.72 ** (i / 2.2)
        st[n:, i % 2] += g * dark[:L - n]
        st[n:, 1 - i % 2] += 0.6 * g * dark[:L - n]
    for c in (0, 1):
        st[:, c] += AIR_G * (st[:, c] - smooth(st[:, c], 8))
    return st

## 10 · Mastering is arithmetic

A track that has not been mastered is not finished, and since **ITU-R BS.1770** there has been no
excuse for treating mastering as a matter of opinion. The recipe: filter the signal with a
K-weighting curve — a high shelf adding 4 dB above about 1.7 kHz, plus a high-pass near 38 Hz,
between them approximating what the ear actually counts as loud — then take the mean square over
400 ms blocks, discard blocks below −70 LUFS absolutely and blocks more than 10 LU below the rest,
and report what survives in LUFS. **EBU Tech 3342** reuses the same machinery over 3-second blocks
and reports the 95th percentile minus the 10th as the loudness range in LU: how far the piece travels
between its quiet and its loud.

Two implementation notes that cost real time to learn. The K-weighting is applied in the **frequency
domain**, because a time-domain recursive filter over 7.5 million samples in interpreted Python is
unusably slow; the two biquads are evaluated analytically as a transfer function on an FFT grid, with
coefficients derived for 44.1 kHz rather than the 48 kHz they are normally tabulated at. And the block
mean square comes off a **cumulative sum**, which turns one pass per block into one pass in total.

Three targets, every one of them published rather than preferred. **−14 LUFS** integrated, because
that is what the streaming platforms normalise to: master quieter and you simply get turned up, master
louder and you get turned down having thrown away your dynamics for nothing. **A loudness range near
5.5 LU**, measured off the reference tracks and what the intensity column was tuned to hit. And a
**peak-to-loudness ratio above 6 dB**, aiming for 10 to 12 — the crest factor that survived the
loudness war, and the difference between music that breathes and a brick. A glue compressor and a
`tanh` soft ceiling get us there with nothing squashed into the rails; the printout below is the
proof, including a count of samples at full scale, which should be zero.

One thing this master deliberately does not do is dither. Rounding a float mix to 16-bit adds
quantisation error, and the textbook fix is to add a tiny amount of shaped noise so that the error
becomes hiss instead of correlated distortion. At a peak of about −2 dBFS the error here sits near
−96 dBFS, roughly forty decibels below the quietest thing in the arrangement, so the cure would be
more audible than the disease. Dither belongs on a 24-bit master being reduced for a quiet classical
release, not here.

In [ ]:
# A biquad's transfer function evaluated on a frequency grid, which turns filtering into a multiply.
def biquad_H(b, a, f, sr):
    z = np.exp(-2j * np.pi * f / sr)
    return (b[0] + b[1] * z + b[2] * z * z) / (a[0] + a[1] * z + a[2] * z * z)


# The BS.1770-4 pre-filter (a high shelf near 1681 Hz) and the RLB high-pass (near 38 Hz), both
# derived for this sample rate rather than assumed at 48 kHz, applied as one frequency-domain
# multiply. The transform is padded past twice the signal length so no filter tail wraps around.
def k_weight(x, sr):
    f0, G, Q = 1681.974450955533, 3.999843853973347, 0.7071752369554196
    K, Vh = math.tan(math.pi * f0 / sr), 10.0 ** (G / 20.0)
    Vb, a0 = Vh ** 0.4996667741545416, 1.0 + K / Q + K * K
    b1 = [(Vh + Vb * K / Q + K * K) / a0, 2.0 * (K * K - Vh) / a0, (Vh - Vb * K / Q + K * K) / a0]
    a1 = [1.0, 2.0 * (K * K - 1.0) / a0, (1.0 - K / Q + K * K) / a0]
    f0, Q = 38.13547087602444, 0.5003270373238773
    K = math.tan(math.pi * f0 / sr)
    d0 = 1.0 + K / Q + K * K
    b2, a2 = [1.0, -2.0, 1.0], [1.0, 2.0 * (K * K - 1.0) / d0, (1.0 - K / Q + K * K) / d0]
    n = x.shape[0]
    nfft = 1 << (int(n - 1).bit_length() + 1)
    f = np.fft.rfftfreq(nfft, 1.0 / sr)
    H = biquad_H(b1, a1, f, sr) * biquad_H(b2, a2, f, sr)
    out = np.empty_like(x)
    for c in range(x.shape[1]):
        out[:, c] = np.fft.irfft(np.fft.rfft(x[:, c], n=nfft) * H, n=nfft)[:n]
    return out


# Gated integrated loudness in LUFS (BS.1770-4) and loudness range in LU (EBU Tech 3342). Both read
# their block energies off one cumulative sum; the channel weights are 1.0 for left and right.
def loudness(x, sr=SR):
    y = k_weight(x, sr)
    cs = np.concatenate([np.zeros((1, y.shape[1])), np.cumsum(y * y, axis=0)])
    def blocks(seconds, hops):
        n = int(round(seconds * sr))
        hop = max(1, n // hops)
        i = np.arange(1 + (cs.shape[0] - 1 - n) // hop) * hop
        z = ((cs[i + n] - cs[i]) / n).sum(axis=1)
        return z, -0.691 + 10.0 * np.log10(np.maximum(z, 1e-20))
    z, l = blocks(0.400, 4)
    keep = l > -70.0                                                   # the absolute gate
    keep &= l > -0.691 + 10.0 * np.log10(z[keep].mean()) - 10.0        # then the relative gate
    integrated = float(-0.691 + 10.0 * np.log10(z[keep].mean()))
    z3, l3 = blocks(3.0, 3)
    k3 = l3 > -70.0
    k3 &= l3 > -0.691 + 10.0 * np.log10(z3[k3].mean()) - 20.0
    return integrated, float(np.percentile(l3[k3], 95) - np.percentile(l3[k3], 10))


# Magnitude spectra of overlapping Hann-windowed frames, in chunks so no enormous array is ever held.
def spectrogram(mono, win=2048, hop=1024):
    n = 1 + (len(mono) - win) // hop
    w = np.hanning(win)
    S = np.empty((n, win // 2 + 1))
    for s in range(0, n, 512):
        k = min(512, n - s)
        idx = (np.arange(s, s + k)[:, None] * hop) + np.arange(win)[None, :]
        S[s:s + k] = np.abs(np.fft.rfft(mono[idx] * w[None, :], axis=1))
    return S, np.fft.rfftfreq(win, 1.0 / SR)


# Median spectral centroid over the frames that carry energy: one number for how bright a mix is.
def centroid(mono):
    S, freqs = spectrogram(mono)
    e = S.sum(axis=1)
    live = e > e.max() * 1e-3
    return float(np.median((S[live] * freqs).sum(axis=1) / np.maximum(e[live], 1e-12)))

### The chain, in order

Mastering is a signal chain and the order is the craft. First **centre and normalise**: remove any DC
offset the summed buses picked up, then scale so the loudest sample sits at exactly one, which gives
every later stage a known input.

Then the **glue compressor**. It reads a peak envelope smoothed over 50 ms, turns anything above a
fifth of full scale down at about 3:1, and smooths the resulting gain curve over another 30 ms so that
it leans on whole loud passages rather than snatching at individual transients. The channels are
linked, because compressing left and right independently makes a hard left-hand hit drag the whole
stereo image to the right.

Then the **soft ceiling**, a plain `tanh`. Below about a third of full scale it is indistinguishable
from a straight wire; above that it bends, so the peaks round off instead of clipping, and the odd
harmonics it adds in the process are a small part of why the mix has any bite at all. A hard clipper
would do the peak-limiting job too, and it would sound like gravel.

Only then the **gain**, and only one measurement is needed to set it. Loudness moves decibel for
decibel with gain, and both of the BS.1770 gates are defined relative to the material's own mean, so
scaling the whole track cannot change which blocks were counted. Measure once, solve for the gain that
lands on −14 LUFS, apply it, done.

In [ ]:
DRIVE, TARGET_LUFS = 1.6, -14.0


# A channel-linked downward compressor: gain reduction taken from a smoothed peak envelope, and then
# the gain curve itself smoothed, so it leans on loud sections instead of pumping on every transient.
# Linking the channels means a hard left-hand transient cannot pull the image toward the right.
def compress(st, thr=0.20, ratio=3.2, attack=0.05, release=0.03):
    env = smooth(np.abs(st).max(axis=1), int(attack * SR))
    g = np.ones_like(env)
    over = env > thr
    g[over] = (env[over] / thr) ** (1.0 / ratio - 1.0)
    return st * smooth(g, int(release * SR))[:, None]


# Centre, normalise, glue-compress, soften the peaks with a tanh, then set the integrated loudness
# exactly. One measurement is enough: loudness moves decibel for decibel with gain, and both of the
# BS.1770 gates are relative, so the final scaling cannot change which blocks were counted.
def master(st):
    st = st - st.mean(axis=0, keepdims=True)
    st = compress(st / max(1e-9, np.abs(st).max()))
    st = np.tanh(st * DRIVE) / math.tanh(DRIVE)
    integrated, lra = loudness(st)
    return st * 10.0 ** ((TARGET_LUFS - integrated) / 20.0), lra


# Weights in, mastered stereo out. Everything between is deterministic, which is the whole reason the
# published weights file can reproduce this exact audio on someone else's machine.
def render_track(model):
    stereo, lra = master(mix(render(compose(model))))
    return stereo[:NS], lra


TRACK, LRA = render_track(MODEL)
PEAK = float(np.abs(TRACK).max())
LUFS = loudness(TRACK)[0]
print(f"integrated loudness {LUFS:7.2f} LUFS  (target {TARGET_LUFS}: the streaming normalisation)")
print(f"loudness range      {LRA:7.2f} LU    (the reference epics measure about 5.5)")
print(f"true peak           {20 * math.log10(PEAK):7.2f} dBFS  peak-to-loudness {20 * math.log10(PEAK) - LUFS:.1f} dB")
print(f"median centroid     {centroid(TRACK.mean(axis=1)):7.0f} Hz    samples at full scale: {int((np.abs(TRACK) >= 0.999).sum())}")

## 11 · The track, and the weights that made it

Three files go out. `out/track.wav` is 44.1 kHz 16-bit stereo PCM written with the standard library's
`wave` module. That is the entire audio toolchain available offline, and it is enough: WAV is
uncompressed, so nothing about the file depends on an encoder's version, its threading, or the day it
was run.

`out/weights.safetensors` is the model, in the Hugging Face **safetensors** layout: eight bytes of
little-endian header length, a UTF-8 JSON header mapping each tensor to its dtype, shape and byte
offsets, then the raw tensor bytes back to back. It is worth writing by hand once to see how little
there is to it, and it is the right container for a published model — readable from any stack in
twenty lines, with no framework required and no code executed on load, which is the whole reason the
format exists.

It also carries a `__metadata__` map of strings, and that is where these two files become one object.
It records `track_sha256`, the SHA-256 of the WAV we have just written, so the weights know exactly
which audio they render; alter one sample of the track, or retrain the model, and the binding breaks
loudly. Note the order: the audio is written first, then hashed, then the hash is stored. And because
the container is assembled with `json` and `struct` rather than `np.savez`, there is no archive
metadata in it of any kind — the same weights always produce the same bytes on disk.

`out/render.json` is the third file, and it is the platform's binding record rather than a
convenience: one small object naming the SHA-256 of the track, the SHA-256 of the weights, the
parameter count, the sample rate and the seed. Every number in it is recomputed by the gate — the
audio is re-hashed, the parameters are counted out of the container — so none of it is a claim that
has to be believed. It also carries `model.source`, which is `"own"` here because these weights were
trained a few cells above; a piece that instead rendered from a model declared at a pinned Hub commit
would write `"hf"`, name that exact revision in `ref`, and ship no weights of its own. One manifest
either way, so a listener never has to guess where the sound came from.

In [ ]:
# Quantise to 16-bit and write interleaved PCM. Returns the samples that actually shipped, so the
# re-render can be compared against the quantised audio rather than against the float mix.
def write_wav(path, stereo):
    # 32767 rather than 32768: scaling by the positive maximum keeps a full-scale sample from wrapping
    # to the opposite rail, and the clip is belt and braces for anything the master let through.
    pcm = np.round(np.clip(stereo, -1.0, 1.0) * 32767.0).astype(np.int16)
    with wave.open(path, "wb") as f:
        f.setnchannels(2)
        f.setsampwidth(2)
        f.setframerate(SR)
        f.writeframes(pcm.tobytes())
    return pcm


# [8-byte little-endian header length][UTF-8 JSON header][raw tensor bytes]. Offsets are relative to
# the end of the header. Names are sorted and the header is padded to a multiple of eight bytes, both
# so that identical weights always serialise to an identical file.
def save_safetensors(path, tensors, meta):
    head, blobs, off = {}, [], 0
    for name in sorted(tensors):
        raw = np.ascontiguousarray(tensors[name], dtype=np.float32).tobytes()
        head[name] = {"dtype": "F32", "shape": list(tensors[name].shape),
                      "data_offsets": [off, off + len(raw)]}
        blobs.append(raw)
        off += len(raw)
    head["__metadata__"] = {k: str(v) for k, v in meta.items()}
    # Sorted keys and the compact separators are what make this byte-stable: two runs that produce the
    # same weights must produce the same header text, down to the spaces that are not there.
    hb = json.dumps(head, sort_keys=True, separators=(",", ":")).encode("utf-8")
    hb += b" " * (-len(hb) % 8)
    with open(path, "wb") as f:
        f.write(struct.pack("<Q", len(hb)) + hb + b"".join(blobs))


PARAMS = int(sum(v.size for v in MODEL.values()))
PCM = write_wav("out/track.wav", TRACK)
TRACK_SHA = hashlib.sha256(open("out/track.wav", "rb").read()).hexdigest()
save_safetensors("out/weights.safetensors", MODEL, {
    "track_sha256": TRACK_SHA, "sample_rate": SR, "seed": SEED,
    "params": PARAMS, "model": "aq-gru-music-1 (31x16 emb, 68-unit GRU)",
    "notebook": "Ten Thousand Dawns", "license": "CC BY 4.0"})

# The binding record, written LAST because it hashes both files that come before it. `source` is
# "own": these weights were trained above, so there is no external model to reference and `ref` stays
# empty; a piece rendering from a declared model would name it there instead and write no weights
# file at all. Sorted keys and a fixed indent keep the bytes identical between two runs, for the same
# reason the container above sorts its header.
WEIGHTS_SHA = hashlib.sha256(open("out/weights.safetensors", "rb").read()).hexdigest()
RENDER = {"track_sha256": TRACK_SHA,
          "model": {"source": "own", "ref": "", "weights_sha256": WEIGHTS_SHA,
                    "params": PARAMS, "sample_rate": SR, "seed": SEED}}
with open("out/render.json", "w") as f:
    json.dump(RENDER, f, indent=1, sort_keys=True)
# Read it back the way the gate reads it, so a typo fails here rather than at review time.
assert json.load(open("out/render.json")) == RENDER, "render.json did not survive the round trip"

print(f"out/track.wav            {os.path.getsize('out/track.wav'):>10,} bytes · {len(PCM) / SR:.2f} s stereo")
print(f"out/weights.safetensors  {os.path.getsize('out/weights.safetensors'):>10,} bytes · {PARAMS:,} parameters")
print(f"out/render.json          {os.path.getsize('out/render.json'):>10,} bytes · binds the two by sha256")
print("track_sha256  ", TRACK_SHA)
print("weights_sha256", WEIGHTS_SHA)

## 12 · The proof: reload the shipped file and render it again

This is the cell that makes the claim checkable rather than merely stated. It forgets the trained
model, reads `out/weights.safetensors` back off disk with nothing but `json`, `struct` and
`np.frombuffer`, runs the entire pipeline again from those numbers — sampling, synthesis, arrangement,
mastering, quantisation — and asserts that the resulting PCM is **byte for byte** what is already in
`out/track.wav`.

If that assertion holds, the audio really is a function of the weights, and anyone who fetches the
file from this work's permanent `/weights` address can render the piece themselves and check the hash
against the one the file carries in its own metadata. If instead the synthesiser were doing the
composing, with a weights file placed beside it for decoration, this is the cell that would fail
first: the notes would come from somewhere the file does not describe.

It is worth being clear about what the assertion does and does not cover. It proves the render is
deterministic and that it depends on nothing but the published weights and the seeds written above.
It does not prove the model is *good* — that is what the falling loss, the sampled grids and your own
ears are for.

In [ ]:
# Read a safetensors container: header length, JSON header, then one view per tensor. Nothing here
# knows about this particular model, which is the point of using a standard container.
def load_safetensors(path):
    with open(path, "rb") as f:
        raw = f.read()
    n = struct.unpack("<Q", raw[:8])[0]
    head = json.loads(raw[8:8 + n].decode("utf-8"))
    meta, body = head.pop("__metadata__", {}), raw[8 + n:]
    out = {}
    for name, t in head.items():
        lo, hi = t["data_offsets"]
        out[name] = np.frombuffer(body[lo:hi], dtype=np.float32).reshape(t["shape"]).astype(np.float64)
    return out, meta


# Three assertions, in the order that a reader should care about them: the file is bound to the audio,
# the numbers survived the round trip through float32, and the audio itself comes back bit for bit.
RELOADED, META = load_safetensors("out/weights.safetensors")
print("tensors read back:", {k: v.shape for k, v in RELOADED.items()})
print("metadata:", {k: META[k] for k in ("sample_rate", "seed", "params", "model")})
assert META["track_sha256"] == TRACK_SHA, "the weights are not bound to the track we shipped"
assert all(np.array_equal(RELOADED[k], MODEL[k]) for k in MODEL), "reloaded weights differ"

REDO, _ = render_track(RELOADED)
REDO_PCM = np.round(np.clip(REDO, -1.0, 1.0) * 32767.0).astype(np.int16)
assert REDO_PCM.tobytes() == PCM.tobytes(), "the re-render does not match out/track.wav"
print(f"re-rendered {REDO_PCM.shape[0]:,} frames from the shipped file: identical to out/track.wav")
print("sha256 of the re-render:", hashlib.sha256(REDO_PCM.tobytes()).hexdigest())

## 13 · Looking at it

Three figures, all drawn in mid-tone ink on a transparent background so that they read on a dark page
and a light one alike. The platform shows work on both, and a figure that only works on one is a
figure half its readers cannot see — which is why the palette here is a mid grey plus the two brand
colours, gold and blue, and nothing else.

The **loss curve** is the training claim, plotted against a dashed line at `ln 31`, the score of a
model that knows nothing — with the held-out curve beside it, because a training curve on its own can
be made to look like anything.

The **envelope** is the arrangement: the peak level of every one of the 64 bars with the section
boundaries marked, and you should be able to find the statement, the two builds and the two drops in it
without being told which is which. The **spectrogram** is the mix, on a log
frequency axis and pooled into quarter-octave bands so that what you see is the balance rather than a
moiré pattern of individual partials: bass and kick along the bottom, hook and choir through the middle,
shimmer and hats up top. The bass arriving at bar 8, the two risers smearing energy across the top
before each drop, and the arrangement thinning over the last eight bars are all visible without being
told where to look.

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Two brand colours and no third accent, each available at full strength and muted toward the neutral
# midpoint the way the platform's contrast engine mutes them. Large filled areas take a muted tint and
# thin lines take more of the full colour, so the average ink of every figure lands near sRGB 127 and
# reads on a dark page and a light one alike. Everything is transparent; nothing paints a background.
INK, YANG, YIN = "#7a7a7a", "#E8B923", "#1746DC"
GOLD_LINE, GOLD_FILL, BLUE_MUTED = "#c9a12e", "#a8862a", "#3f5aa8"
mpl.rcParams.update({"figure.facecolor": "none", "axes.facecolor": "none", "savefig.transparent": True,
                     "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": INK,
                     "xtick.color": INK, "ytick.color": INK, "font.size": 9, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
DUAL = LinearSegmentedColormap.from_list("aq", [BLUE_MUTED, "#5f6b88", "#767676", "#84795f",
                                               GOLD_FILL, "#d1a620"])


# Save every figure transparently beside the track, then draw it inline.
def show(fig, name):
    # Bounding-box trimming keeps the transparent margin from dominating, and the figures land in out/
    # beside the track, so they publish to the same address as the audio they describe.
    fig.savefig(f"out/{name}.png", transparent=True, bbox_inches="tight")
    plt.show()


fig, ax = plt.subplots(figsize=(7.2, 2.6))
ax.plot(np.arange(1, len(HISTORY) + 1), HISTORY, color=GOLD_LINE, lw=1.4, label="training batches")
ax.plot([c[0] for c in CHECKS], [c[1] for c in CHECKS], color=YIN, lw=1.3, marker="o", ms=2.6,
        label="held-out phrases")
ax.axhline(math.log(VOCAB), color=INK, lw=1.1, ls="--")
ax.text(len(HISTORY) * 0.6, math.log(VOCAB) + 0.09, "uniform guessing: ln 31", color=INK, fontsize=8)
ax.legend(frameon=False, fontsize=8, labelcolor=INK)
ax.set(xlabel="Adam update", ylabel="cross-entropy (nats/token)", title="Training the sequence model")
show(fig, "figure-loss")

bars = np.abs(TRACK).max(axis=1).reshape(BARS, BAR).max(axis=1)
fig, ax = plt.subplots(figsize=(7.2, 2.6))
ax.bar(np.arange(BARS), bars, width=0.82, color=GOLD_FILL, edgecolor="none")
for s0, s1, name, *_ in FORM:
    ax.axvline(s0 - 0.5, color=YIN, lw=1.0)
    ax.text(s0 + 0.2, bars.max() * 1.03, name, color=INK, fontsize=7.5, ha="left")
ax.set(xlabel="bar", ylabel="peak level", title="The arrangement, bar by bar", ylim=(0, bars.max() * 1.17))
show(fig, "figure-envelope")

# Additive synthesis puts a narrow comb of partials in every frame, and a linear-frequency picture of
# that comb turns to moire the moment it is squeezed into a few hundred pixels. So the spectra are
# pooled into 48 log-spaced bands — a quarter-octave each, which is how a musician hears frequency —
# and averaged 28 frames at a time, giving one column per 1.3 s. Both poolings are means, so the
# picture still shows real energy rather than whichever partial happened to land on a pixel.
S, freqs = spectrogram(TRACK.mean(axis=1), 2048, 2048)
NB, GROUP = 48, 28
edges = np.geomspace(40.0, 16000.0, NB + 1)
bands = np.empty((S.shape[0], NB))
for b in range(NB):
    sel = (freqs >= edges[b]) & (freqs < edges[b + 1])
    bands[:, b] = S[:, sel].mean(axis=1) if sel.any() else S[:, np.argmin(np.abs(freqs - edges[b]))]
tiles = bands[:S.shape[0] // GROUP * GROUP].reshape(-1, GROUP, NB).mean(axis=1)
dB = 20.0 * np.log10(np.maximum(tiles.T, 1e-6))
fig, ax = plt.subplots(figsize=(7.2, 3.0))
ax.imshow(dB, origin="lower", aspect="auto", cmap=DUAL, extent=[0, NS / SR, 0, NB],
          vmin=np.percentile(dB, 10), vmax=np.percentile(dB, 99.9))
ax.set_yticks([float(np.searchsorted(edges, k)) for k in (100, 400, 1600, 6400)], ["0.1", "0.4", "1.6", "6.4"])
ax.set(xlabel="seconds", ylabel="kHz (log)", title="Spectrogram of the master")
show(fig, "figure-spectrogram")

## 14 · What you can do with this

**Change the music.** The corpus grammar in section 2 is where the style lives. Swap `SCALE` for
Phrygian, `(0, 1, 3, 5, 7, 8, 10)`, and the piece turns Middle Eastern without another edit. Change
`SPLIT_P` and the rhythm loosens or tightens; raise `rest_p` and the phrasing starts to breathe.
Retrain, and the model will have learned a different idiom. The `FORM` table in section 8 is the
arrangement — reorder the sections, transpose the return somewhere else, take the drums out of the
second build, put the break first. Everything downstream still holds, and the mastering cell will tell
you in LUFS and LU whether it still holds together *well*.

**Re-render the weights.** `out/weights.safetensors` publishes to this work's permanent `/weights`
address beside its DOI: 80 kB carrying every number the piece is made of. Fetch it, read it with the
loader in section 12, call `render_track` on it, and compare the SHA-256 of what comes out with
the one the file carries in its own metadata. If they match, you have reproduced a piece of music from
its model — which is the only definition of reproducible audio that survives contact with a sandbox
that has no network, no sample library and no encoder.

**Push on the honest limits.** This model is a single-layer GRU with twenty thousand parameters
trained on a grammar someone wrote in an afternoon: it can hold a phrase together and return to a
motif, and it cannot plan a modulation or invent a rhythm the grammar never showed it. The harmony
under each section is the model's own choice; the eight-section arc around it is not. A bigger model,
a corpus with more than one idiom in it, and a form the model chooses for itself are all sitting
right there, and every one of them can be attempted in this notebook without leaving numpy.